# Lab 02 – Model Monitoring & Drift Alerts

**Scenario:** Production readiness requires continuous visibility into model behavior, data freshness, and guardrail effectiveness. You will configure telemetry pipelines, detect drift, and route alerts to incident tooling.

## Objectives
- Instrument Langfuse/Prometheus metrics needed for Week 9 SLOs
- Configure drift detection jobs for input and output signals
- Send alerts to PagerDuty/Slack when thresholds breach
- Update readiness scorecard with monitoring coverage

In [ ]:
from pathlib import Path

expected_files = [
    Path('observability/tracing/config.yaml'),
    Path('observability/metrics/rules.yaml'),
    Path('evaluations/drift/baseline.json'),
]
missing = [str(p) for p in expected_files if not p.exists()]
if missing:
    raise FileNotFoundError(f'Provision monitoring assets before proceeding: {missing}')

print('Monitoring assets detected. Proceed with configuration.')

## Step 1 – Telemetry Hook Validation
Verify that traces emit guardrail verdicts, persona attributes, and latency buckets. Extend Prometheus rules to cover SLO targets.

In [ ]:
import json

sample_trace = Path('observability/tracing/sample-trace.json')
if not sample_trace.exists():
    raise FileNotFoundError('Provide a sample Langfuse trace export for validation.')

payload = json.loads(sample_trace.read_text())
keys = {k for span in payload.get('spans', []) for k in span.get('attributes', {})}
required_keys = {'persona', 'guardrail_decision', 'fallback_used', 'latency_ms'}
missing_keys = required_keys - keys
if missing_keys:
    raise KeyError(f'Missing span attributes: {missing_keys}')

print('Trace payload contains required attributes for SLO tracking.')

### Prometheus Rule Patch
Enhance the metrics rule set to emit alerts for latency, success rate, guardrail block rate, and drift scores.

In [ ]:
from textwrap import dedent

rules_path = Path('observability/metrics/rules.yaml')
rules_path.parent.mkdir(parents=True, exist_ok=True)
rules_path.write_text(dedent('''
groups:
  - name: genai-slo-rules
    rules:
      - alert: GenAIHighLatency
        expr: histogram_quantile(0.95, sum(rate(http_request_latency_ms_bucket{service="assistant-api"}[5m])) by (le)) > 2000
        for: 10m
        labels:
          severity: sev2
        annotations:
          summary: p95 latency exceeds 2s
      - alert: GenAISuccessDrop
        expr: sum(rate(http_requests_total{service="assistant-api",status="success"}[5m]))
              / sum(rate(http_requests_total{service="assistant-api"}[5m])) < 0.97
        for: 5m
        labels:
          severity: sev2
        annotations:
          summary: Success rate below 97%
      - alert: GuardrailBlockRateLow
        expr: sum(rate(guardrail_blocks_total{decision="block"}[30m]))
              / sum(rate(guardrail_evaluations_total[30m])) < 0.93
        for: 30m
        labels:
          severity: sev2
        annotations:
          summary: Guardrail block rate below threshold
      - alert: OutputDriftDetected
        expr: genai_output_drift_score > 0.25
        for: 15m
        labels:
          severity: sev2
        annotations:
          summary: Output drift score exceeds threshold
'''))
print(f'Wrote Prometheus rule configuration to {rules_path}')

## Step 2 – Configure Drift Detection
Calibrate baseline windows, thresholds, and alert routing using the evaluation harness.

In [ ]:
import datetime as dt

from evaluations.drift import DriftMonitor
from notifications import PagerDutyClient

monitor = DriftMonitor(
    baseline_window=dt.timedelta(days=14),
    comparison_window=dt.timedelta(hours=6),
    signals={
        'embedding_distance': 0.25,
        'helpfulness_score_delta': -0.4,
        'guardrail_bypass_rate': 0.02,
    },
)

result = monitor.run()
snapshot_dir = Path('artifacts/drift')
snapshot_dir.mkdir(parents=True, exist_ok=True)
monitor.snapshot(snapshot_dir)

if result.exceeds_thresholds:
    PagerDutyClient().trigger(
        routing_key='PD_ROUTING_KEY',
        severity='error',
        summary='GenAI output drift detected',
        details=result.to_dict(),
    )
    print('Alert dispatched to PagerDuty.')
else:
    print('No drift detected. Continue monitoring.')

## Step 3 – Alert Routing Test
Trigger synthetic alerts to validate Slack/PagerDuty integration and verify runbooks are linked.

In [ ]:
from incident.alerts import SlackWebhook

payload = {
    'severity': 'sev2',
    'title': 'Synthetic drift alert',
    'runbook': 'resources/incident-playbook-template.md',
    'slo': 'output_quality',
}

SlackWebhook(url='https://hooks.slack.com/services/T000/B000/XXXXX').send(payload)
print('Posted synthetic alert. Capture screenshot for evidence.')

## Step 4 – Update Scorecard
Complete the monitoring rows in `resources/production-slo-scorecard.csv` and capture links to dashboards, alert policies, and drift jobs.

## Retrospective & Submission Checklist
- Langfuse trace screenshot showing guardrail attributes
- Prometheus/Grafana dashboard with new alerts
- Drift monitor report stored in `artifacts/drift/`
- Alert routing test evidence (Slack screenshot, PagerDuty incident link)
- Scorecard updated with status and owners